In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to build summary row for a dataframe
def summarize_df(df, file_label, name_col=None, message_only=False):
    row_count = len(df)
    if row_count == 0:
        min_epoch = max_epoch = None
        min_local = max_local = None
    else:
        min_epoch = int(df['timestamp'].min())
        max_epoch = int(df['timestamp'].max())
        min_local = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        max_local = pd.to_datetime(max_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    # sample cmdb ids (up to 20)
    sample_cmdb = []
    if 'cmdb_id' in df.columns:
        sample_cmdb = list(pd.Series(df['cmdb_id'].dropna().unique())[:20])
    # sample names/messages (up to 20)
    sample_names = []
    if message_only:
        if 'message' in df.columns:
            sample_names = list(pd.Series(df['message'].dropna().unique())[:20])
    else:
        if name_col and name_col in df.columns:
            sample_names = list(pd.Series(df[name_col].dropna().unique())[:20])
    # compact join for display
    sample_cmdb_str = '; '.join(map(str, sample_cmdb))
    sample_names_str = '; '.join(map(str, sample_names))
    return {
        'file': file_label,
        'rows': row_count,
        'min_epoch': min_epoch,
        'max_epoch': max_epoch,
        'min_local': min_local,
        'max_local': max_local,
        'sample_cmdb_ids': sample_cmdb_str,
        'sample_names_or_messages': sample_names_str
    }

metric_summary = summarize_df(metric_df, 'metric.csv', name_col='kpi_name', message_only=False)
trace_summary  = summarize_df(trace_df,  'trace.csv',  name_col='trace_name', message_only=False)
log_summary    = summarize_df(log_df,    'log.csv',    name_col='log_name',   message_only=False)
error_summary  = summarize_df(error_df,  'error_logs.csv', name_col=None,    message_only=True)

summary_df = pd.DataFrame([metric_summary, trace_summary, log_summary, error_summary])

# Prepare top-5 compact tables for each file showing only required columns
def top5_view(df, name_col=None, value_col='value', message_only=False):
    if message_only:
        cols = ['timestamp', 'cmdb_id', 'message']
    else:
        cols = ['timestamp', 'cmdb_id']
        if name_col:
            cols.append(name_col)
        if value_col and value_col in df.columns:
            cols.append(value_col)
    # guard missing columns
    cols = [c for c in cols if c in df.columns]
    top5 = df.loc[:, cols].head(5).copy()
    # add local readable timestamp column (compact)
    if 'timestamp' in top5.columns:
        top5['timestamp_local'] = pd.to_datetime(top5['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S')
        # keep timestamp as epoch as well but show local near-by
    return top5

metric_top5 = top5_view(metric_df, name_col='kpi_name', value_col='value', message_only=False)
trace_top5  = top5_view(trace_df,  name_col='trace_name', value_col='value', message_only=False)
log_top5    = top5_view(log_df,    name_col='log_name',   value_col='value', message_only=False)
error_top5  = top5_view(error_df,  name_col=None,         value_col=None,     message_only=True)

# Display compact summary and top5 for each file (variables to be rendered by IPython)
summary_df, metric_top5, trace_top5, log_top5, error_top5

```
Out[1]:
```
```python
# Summarize the earlier execution results into a concise plain-English summary string.
# Reuse variables already loaded in the IPython state: summary_df, metric_top5, trace_top5, log_top5, error_top5.

# Helper to fetch row from summary_df by filename
def row_for(file_name):
    return summary_df[summary_df['file'] == file_name].iloc[0]

m = row_for('metric.csv')
t = row_for('trace.csv')
l = row_for('log.csv')
e = row_for('error_logs.csv')

def top_example(df, name_col):
    if df.shape[0] == 0:
        return "(no rows)"
    r = df.iloc[0].to_dict()
    # Build compact example string showing epoch, local ts, cmdb_id, name/message and value
    epoch = int(r.get('timestamp', ''))
    local = r.get('timestamp_local', '')
    cmdb = r.get('cmdb_id', '')
    if 'message' in r:
        detail = r.get('message', '')
    else:
        name = r.get(name_col, '') if name_col else ''
        val = r.get('value', '')
        detail = f"{name} = {val}"
    return f"{epoch} ({local}) | {cmdb} | {detail}"

metric_example = top_example(metric_top5, 'kpi_name')
trace_example  = top_example(trace_top5,  'trace_name')
log_example    = top_example(log_top5,    'log_name')
error_example  = top_example(error_top5,  None)

summary = (
    "Compact telemetry summary (times shown in Asia/Shanghai / UTC+8):\n\n"
    f"metric.csv: {m['rows']} rows, time range {m['min_local']} to {m['max_local']}. "
    f"Sample cmdb_ids (up to 20): {m['sample_cmdb_ids']}. "
    f"Sample kpi_names (up to 20): {m['sample_names_or_messages']}. "
    f"Top-row example: {metric_example}\n\n"
    f"trace.csv: {t['rows']} rows, time range {t['min_local']} to {t['max_local']}. "
    f"Sample cmdb_ids (up to 20): {t['sample_cmdb_ids']}. "
    f"Sample trace_names (up to 20): {t['sample_names_or_messages']}. "
    f"Top-row example: {trace_example}\n\n"
    f"log.csv: {l['rows']} rows, time range {l['min_local']} to {l['max_local']}. "
    f"Sample cmdb_ids (up to 20): {l['sample_cmdb_ids']}. "
    f"Sample log_names (up to 20): {l['sample_names_or_messages']}. "
    f"Top-row example: {log_example}\n\n"
    f"error_logs.csv: {e['rows']} rows, time range {e['min_local']} to {e['max_local']}. "
    f"Sample cmdb_ids (up to 20): {e['sample_cmdb_ids']}. "
    f"Sample messages (up to 20): {e['sample_names_or_messages']}. "
    f"Top-row example: {error_example}\n\n"
    "Notes: time window spans 2022-03-21 07:30:00 to ~07:59:56 CST (UTC+8). "
    "Metric data is large (147,732 rows); trace and log sets are smaller. Error logs show repeated warnings from frontend pods."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file    rows   min_epoch   max_epoch                min_local                max_local                                    sample_cmdb_ids                           sample_names_or_messages
0      metric.csv  147732  1647819000  1647820740  2022-03-21 07:30:00 CST  2022-03-21 07:59:00 CST  adservice; adservice-0; adservice-1; adservice...  app.grpc.count; app.grpc.mrt; app.grpc.rr; app...
1       trace.csv   25480  1647819000  1647820740  2022-03-21 07:30:00 CST  2022-03-21 07:59:00 CST  adservice-0; adservice-1; adservice-2; adservi...  trace.from_frontend-0.duration_mean; trace.fro...
2         log.csv    1568  1647819000  1647820740  2022-03-21 07:30:00 CST  2022-03-21 07:59:00 CST  adservice-0; adservice-1; adservice-2; cartser...                     log.error_count; log.row_count
3  error_logs.csv     309  1647819005  1647820796  2022-03-21 07:30:05 CST  2022-03-21 07:59:56 CST                 frontend-0; frontend-2; frontend-1  severity: warning, message: failed to retrieve...,     timestamp    cmdb_id        kpi_name       value      timestamp_local
0  1647819000  adservice  app.grpc.count   28.000000  2022-03-21 07:30:00
1  1647819000  adservice    app.grpc.mrt    0.002695  2022-03-21 07:30:00
2  1647819000  adservice     app.grpc.rr  100.000000  2022-03-21 07:30:00
3  1647819000  adservice     app.grpc.sr   85.714286  2022-03-21 07:30:00
4  1647819000  adservice  app.http.count   24.000000  2022-03-21 07:30:00,     timestamp      cmdb_id                           trace_name     value      timestamp_local
0  1647819000  adservice-0  trace.from_frontend-0.duration_mean  0.000015  2022-03-21 07:30:00
1  1647819000  adservice-0   trace.from_frontend-0.duration_p95  0.000016  2022-03-21 07:30:00
2  1647819000  adservice-0     trace.from_frontend-0.error_rate  0.000000  2022-03-21 07:30:00
3  1647819000  adservice-0      trace.from_frontend-0.row_count  3.000000  2022-03-21 07:30:00
4  1647819000  adservice-0  trace.from_frontend-1.duration_mean  0.000015  2022-03-21 07:30:00,     timestamp      cmdb_id         log_name  value      timestamp_local
0  1647819000  adservice-0  log.error_count    0.0  2022-03-21 07:30:00
1  1647819000  adservice-0    log.row_count  100.0  2022-03-21 07:30:00
2  1647819000  adservice-1  log.error_count    0.0  2022-03-21 07:30:00
3  1647819000  adservice-1    log.row_count    6.0  2022-03-21 07:30:00
4  1647819000  adservice-2  log.error_count    0.0  2022-03-21 07:30:00,     timestamp     cmdb_id                                            message      timestamp_local
0  1647819005  frontend-0  severity: warning, message: failed to retrieve...  2022-03-21 07:30:05
1  1647819009  frontend-0  severity: warning, message: failed to retrieve...  2022-03-21 07:30:09
2  1647819018  frontend-0  severity: warning, message: failed to retrieve...  2022-03-21 07:30:18
3  1647819026  frontend-0  severity: warning, message: failed to retrieve...  2022-03-21 07:30:26
4  1647819033  frontend-0  severity: warning, message: failed to retrieve...  2022-03-21 07:30:33)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df loaded earlier
# ensure timestamp is integer
metric_df['timestamp'] = metric_df['timestamp'].astype(int)

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for every (cmdb_id, kpi_name) across the full series
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count_points='count',
         global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 =lambda s: float(s.quantile(0.05)))
    .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'global_p5', 'count_points']]
)

# 2) Filter metric.csv to the incident window [1647819000, 1647820800)
start_ts = 1647819000
end_ts = 1647820800
metric_incident = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# 3) Join thresholds to incident rows to evaluate anomalies by global thresholds
metric_incident = metric_incident.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies using global thresholds
metric_incident['is_high_p95'] = metric_incident['value'] >= metric_incident['global_p95']
metric_incident['is_low_p5']  = metric_incident['value'] <= metric_incident['global_p5']
metric_incident['is_any_anom'] = metric_incident['is_high_p95'] | metric_incident['is_low_p5']

# helper column for earliest anomaly epoch (only keep timestamp where anomaly else NaN)
metric_incident['anomaly_epoch'] = np.where(metric_incident['is_any_anom'], metric_incident['timestamp'], np.nan)

# aggregate per (cmdb_id, kpi_name) within incident window
agg = metric_incident.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    anomaly_count_p95_high = ('is_high_p95', 'sum'),
    anomaly_count_p5_low   = ('is_low_p5', 'sum'),
    total_points_incident  = ('value', 'count'),
    max_value_incident     = ('value', 'max'),
    min_value_incident     = ('value', 'min'),
    earliest_anomaly_epoch = ('anomaly_epoch', 'min')
)

# convert earliest_anomaly_epoch from float NaN to Int where present, and add local string
def epoch_to_local_str(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg['earliest_anomaly_epoch'] = agg['earliest_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg['earliest_anomaly_local'] = agg['earliest_anomaly_epoch'].apply(lambda v: epoch_to_local_str(v) if not pd.isna(v) else '')

# compute total_anomalies as sum of two counts
agg['total_anomalies'] = agg['anomaly_count_p95_high'] + agg['anomaly_count_p5_low']

# join global thresholds for final summary (some groups may lack thresholds if they didn't exist, but thresholds were computed from full series so should exist)
metric_anomaly_summary = agg.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# reorder/select desired columns
metric_anomaly_summary = metric_anomaly_summary[[
    'cmdb_id', 'kpi_name',
    'anomaly_count_p95_high', 'anomaly_count_p5_low', 'total_anomalies',
    'earliest_anomaly_epoch', 'earliest_anomaly_local',
    'max_value_incident', 'min_value_incident',
    'global_p95', 'global_p5'
]]

# 4) Prepare compact top-20 table sorted by total_anomalies desc
metric_anomaly_summary_top20 = metric_anomaly_summary.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# total number of distinct (cmdb_id, kpi_name) pairs scanned within the incident window
total_pairs_scanned = metric_incident.groupby(['cmdb_id', 'kpi_name']).ngroups

# expose variables for later steps: metric_thresholds, metric_incident, metric_anomaly_summary, metric_anomaly_summary_top20, total_pairs_scanned
metric_thresholds.shape, metric_incident.shape, metric_anomaly_summary.shape, metric_anomaly_summary_top20, total_pairs_scanned

```
Out[2]:
```
```python
# Compose a concise plain-English summary using existing kernel variables.
n_pairs = total_pairs_scanned
n_incident_rows = metric_incident.shape[0]
top20 = metric_anomaly_summary_top20

top_first = top20.iloc[0] if not top20.empty else None

summary = (
    f"Metric anomaly summary (incident window 2022-03-21 07:30:00 to 08:00:00 CST / UTC+8):\n\n"
    f"- Global thresholds computed for {n_pairs} distinct (cmdb_id, kpi_name) pairs.\n"
    f"- Incident-window metric rows: {n_incident_rows} rows.\n"
)

if top_first is not None:
    summary += (
        f"- Top anomalies are dominated by cmdb_id '{top_first['cmdb_id']}'.\n"
        f"  Example top KPI: '{top_first['kpi_name']}' has {int(top_first['total_anomalies'])} total anomalies "
        f"({int(top_first['anomaly_count_p95_high'])} >= global P95, {int(top_first['anomaly_count_p5_low'])} <= global P5). "
        f"Earliest anomaly at {top_first['earliest_anomaly_local']}.\n"
    )

summary += (
    "- In the top-20 results many KPIs for 'shippingservice2-0' (container.node-5.*) show the highest counts —\n"
    "  several entries report 60 anomalies (30 high and 30 low) with earliest occurrence at the start of the window.\n"
    "- Note: thresholds were computed globally using the entire series before filtering; anomaly counts reflect values\n"
    "  compared against those global P95/P5 thresholds within the incident window.\n\n"
    "You can inspect 'metric_thresholds', 'metric_incident', and 'metric_anomaly_summary' variables for details,\n"
    "and 'metric_anomaly_summary_top20' for the compact top-20 table."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((4934, 5), (147732, 11), (4934, 11),                cmdb_id                                           kpi_name  anomaly_count_p95_high  anomaly_count_p5_low  total_anomalies  earliest_anomaly_epoch earliest_anomaly_local  max_value_incident  min_value_incident    global_p95     global_p5
0   shippingservice2-0     container.node-5.container_memory_max_usage_MB                      30                    30               60              1647819000    2022-03-21 07:30:00        1.280117e+02        1.280117e+02  1.280117e+02  1.280117e+02
1   shippingservice2-0             container.node-5.container_memory_swap                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
2   shippingservice2-0  container.node-5.container_network_receive_err...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
3   shippingservice2-0  container.node-5.container_fs_writes_merged./d...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
4   shippingservice2-0            container.node-5.container_memory_cache                      30                    30               60              1647819000    2022-03-21 07:30:00        1.633485e+07        1.633485e+07  1.633485e+07  1.633485e+07
5   shippingservice2-0          container.node-5.container_memory_failcnt                      30                    30               60              1647819000    2022-03-21 07:30:00        2.397029e+06        2.397029e+06  2.397029e+06  2.397029e+06
6   shippingservice2-0  container.node-5.container_memory_failures.con...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
7   shippingservice2-0  container.node-5.container_memory_failures.hie...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
8   shippingservice2-0  container.node-5.container_fs_reads_merged./de...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
9   shippingservice2-0  container.node-5.container_fs_sector_reads./de...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
10  shippingservice2-0  container.node-5.container_fs_sector_writes./d...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
11  shippingservice2-0  container.node-5.container_fs_write_seconds./d...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
12  shippingservice2-0      container.node-5.container_fs_writes./dev/vda                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
13  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
14  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
15  shippingservice2-0  container.node-5.container_fs_io_current./dev/...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
16  shippingservice2-0  container.node-5.container_fs_io_time_seconds....                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
17  shippingservice2-0  container.node-5.container_fs_io_time_weighted...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00
18  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1                      30                    30               60              1647819000    2022-03-21 07:30:00        6.046307e+05        6.046307e+05  6.046307e+05  6.046307e+05
19  shippingservice2-0  container.node-5.container_fs_read_seconds./de...                      30                    30               60              1647819000    2022-03-21 07:30:00        0.000000e+00        0.000000e+00  0.000000e+00  0.000000e+00, 4934)```
```

In [ ]:
# In[3]:

import numpy as np
# Reuse trace_df and tz from kernel state

# Ensure timestamp is integer
trace_df['timestamp'] = trace_df['timestamp'].astype(int)

# 1) Compute global P95 and P5 for every (cmdb_id, trace_name) across the full series
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count_points='count',
         global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 =lambda s: float(s.quantile(0.05)))
    .loc[:, ['cmdb_id', 'trace_name', 'global_p95', 'global_p5', 'count_points']]
)

# 2) Filter trace.csv to the incident window [1647819000, 1647820800)
start_ts = 1647819000
end_ts = 1647820800
trace_incident = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)].copy()

# 3) Join thresholds to incident rows and mark anomalies using global thresholds
trace_incident = trace_incident.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

trace_incident['is_high_p95'] = trace_incident['value'] >= trace_incident['global_p95']
trace_incident['is_low_p5']  = trace_incident['value'] <= trace_incident['global_p5']
trace_incident['is_any_anom'] = trace_incident['is_high_p95'] | trace_incident['is_low_p5']
trace_incident['anomaly_epoch'] = np.where(trace_incident['is_any_anom'], trace_incident['timestamp'], np.nan)

# Aggregate per (cmdb_id, trace_name) within the incident window
agg = trace_incident.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    anomaly_count_p95_high = ('is_high_p95', 'sum'),
    anomaly_count_p5_low   = ('is_low_p5', 'sum'),
    total_points_incident  = ('value', 'count'),
    max_value_incident     = ('value', 'max'),
    min_value_incident     = ('value', 'min'),
    earliest_anomaly_epoch = ('anomaly_epoch', 'min')
)

# convert earliest_anomaly_epoch and add local string (reuse epoch_to_local_str if present)
def epoch_to_local_str_safe(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg['earliest_anomaly_epoch'] = agg['earliest_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg['earliest_anomaly_local'] = agg['earliest_anomaly_epoch'].apply(lambda v: epoch_to_local_str_safe(v) if not pd.isna(v) else '')

agg['total_anomalies'] = agg['anomaly_count_p95_high'] + agg['anomaly_count_p5_low']

# 4) Prepare final summary by joining global thresholds for completeness
trace_anomaly_summary = agg.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

trace_anomaly_summary = trace_anomaly_summary[[
    'cmdb_id', 'trace_name',
    'anomaly_count_p95_high', 'anomaly_count_p5_low', 'total_anomalies',
    'earliest_anomaly_epoch', 'earliest_anomaly_local',
    'max_value_incident', 'min_value_incident',
    'global_p95', 'global_p5'
]]

# Top-20 sorted by total_anomalies desc
trace_anomaly_summary_top20 = trace_anomaly_summary.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Total distinct pairs scanned within incident window
total_trace_pairs_scanned = trace_incident.groupby(['cmdb_id', 'trace_name']).ngroups

# Expose compact outputs for inspection
trace_thresholds.shape, trace_incident.shape, trace_anomaly_summary.shape, trace_anomaly_summary_top20, total_trace_pairs_scanned

```
Out[3]:
```
```python
# Build a concise plain-English summary using existing kernel variables:
# trace_thresholds, trace_incident, trace_anomaly_summary_top20, total_trace_pairs_scanned

n_pairs = int(total_trace_pairs_scanned)
n_thresholds = int(trace_thresholds.shape[0])
n_incident_rows = int(trace_incident.shape[0])

top20 = trace_anomaly_summary_top20
top_first = top20.iloc[0] if not top20.empty else None

summary_lines = [
    "Trace anomaly summary (incident window 2022-03-21 07:30:00 to 08:00:00 CST / UTC+8):",
    f"- Global thresholds computed for {n_thresholds} (cmdb_id, trace_name) pairs; {n_pairs} distinct pairs were scanned in the incident window.",
    f"- Incident-window trace rows: {n_incident_rows} rows.",
]

if top_first is not None:
    summary_lines.append(
        f"- Top anomaly example: cmdb_id='{top_first['cmdb_id']}', trace_name='{top_first['trace_name']}'. "
        f"total_anomalies={int(top_first['total_anomalies'])} "
        f"({int(top_first['anomaly_count_p95_high'])} >= global_p95, {int(top_first['anomaly_count_p5_low'])} <= global_p5). "
        f"Earliest anomaly at {top_first['earliest_anomaly_local']}."
    )

summary_lines.append(
    "- Observation: many top entries are error-rate or related trace KPIs (e.g., trace.*.error_rate) "
    "that show global_p95 and global_p5 equal to 0.0. Because anomaly checks use >= global_p95 and <= global_p5, "
    "data points equal to 0.0 meet both conditions and are counted in both high and low anomaly counters—"
    "this doubles the reported total_anomalies for those series."
)

summary_lines.append(
    "- Recommendation: inspect trace_thresholds and example time series in 'trace_incident' for these KPIs. "
    "If the dual-counting behavior is undesired, adjust anomaly logic (use strict > for high or < for low, "
    "or handle equal-threshold cases)."
)

summary = "\n\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((1240, 5), (25480, 11), (1240, 11),                cmdb_id                                trace_name  anomaly_count_p95_high  anomaly_count_p5_low  total_anomalies  earliest_anomaly_epoch earliest_anomaly_local  max_value_incident  min_value_incident  global_p95  global_p5
0   shippingservice2-0         trace.from_frontend2-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
1          adservice-1          trace.from_frontend-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
2          adservice-0          trace.from_frontend-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
3          adservice-2          trace.from_frontend-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
4   shippingservice2-0  trace.from_checkoutservice2-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
5          adservice-1          trace.from_frontend-2.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
6      emailservice2-0  trace.from_checkoutservice2-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
7          adservice-0          trace.from_frontend-1.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
8           frontend-0           trace.to_adservice-1.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
9           frontend-0           trace.to_adservice-2.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
10          frontend-0         trace.to_cartservice-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
11          frontend-0         trace.to_cartservice-1.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
12          frontend-0           trace.to_adservice-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
13       cartservice-1                   trace.self.duration_p95                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
14       cartservice-1       trace.from_frontend-2.duration_mean                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
15       cartservice-1        trace.from_frontend-2.duration_p95                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
16       cartservice-1          trace.from_frontend-2.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
17       cartservice-1          trace.from_frontend-1.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
18          frontend-0     trace.to_currencyservice-0.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
19          frontend-0     trace.to_currencyservice-2.error_rate                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0, 1240)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes: log_df, error_df, tz
# Ensure timestamp integer
log_df['timestamp'] = log_df['timestamp'].astype(int)
error_df['timestamp'] = error_df['timestamp'].astype(int)

# 1) Compute global P95 and P5 for every (cmdb_id, log_name)
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count_points='count',
         global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 =lambda s: float(s.quantile(0.05)))
    .loc[:, ['cmdb_id', 'log_name', 'global_p95', 'global_p5', 'count_points']]
)

# 2) Filter to incident window [1647819000, 1647820800)
start_ts = 1647819000
end_ts = 1647820800
log_incident = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] < end_ts)].copy()

# 3) Join thresholds and compute anomalies using global thresholds
log_incident = log_incident.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

log_incident['is_high_p95'] = log_incident['value'] >= log_incident['global_p95']
log_incident['is_low_p5']  = log_incident['value'] <= log_incident['global_p5']
log_incident['is_any_anom'] = log_incident['is_high_p95'] | log_incident['is_low_p5']
log_incident['anomaly_epoch'] = np.where(log_incident['is_any_anom'], log_incident['timestamp'], np.nan)

# Aggregate per (cmdb_id, log_name) within incident window
agg_log = log_incident.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    anomaly_count_p95_high = ('is_high_p95', 'sum'),
    anomaly_count_p5_low   = ('is_low_p5', 'sum'),
    total_points_incident  = ('value', 'count'),
    max_value_incident     = ('value', 'max'),
    min_value_incident     = ('value', 'min'),
    earliest_anomaly_epoch = ('anomaly_epoch', 'min')
)

# convert earliest_anomaly_epoch and add local string
tz = pytz.timezone('Asia/Shanghai')
def epoch_to_local_str_safe(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg_log['earliest_anomaly_epoch'] = agg_log['earliest_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg_log['earliest_anomaly_local'] = agg_log['earliest_anomaly_epoch'].apply(lambda v: epoch_to_local_str_safe(v) if not pd.isna(v) else '')

agg_log['total_anomalies'] = agg_log['anomaly_count_p95_high'] + agg_log['anomaly_count_p5_low']

# 4) Final summary: join thresholds and select requested columns
log_anomaly_summary = agg_log.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

log_anomaly_summary = log_anomaly_summary[[
    'cmdb_id', 'log_name',
    'anomaly_count_p95_high', 'anomaly_count_p5_low', 'total_anomalies',
    'earliest_anomaly_epoch', 'earliest_anomaly_local',
    'max_value_incident', 'min_value_incident',
    'global_p95', 'global_p5'
]]

log_anomaly_summary_top20 = log_anomaly_summary.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

total_log_pairs_scanned = log_incident.groupby(['cmdb_id', 'log_name']).ngroups
log_incident_rows = log_incident.shape[0]

# 5) error_logs.csv: aggregate counts per cmdb_id within the incident window and prepare top-5 summary
error_incident = error_df[(error_df['timestamp'] >= start_ts) & (error_df['timestamp'] < end_ts)].copy()

# group by cmdb_id
err_grp = (
    error_incident
    .groupby('cmdb_id')
    .agg(count=('timestamp', 'count'),
         earliest_error_epoch=('timestamp', 'min'))
    .reset_index()
)

# add earliest local time
err_grp['earliest_error_local'] = err_grp['earliest_error_epoch'].apply(lambda v: pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S'))

# collect up to 5 distinct sample messages per cmdb_id
def sample_messages_for(cmdb):
    msgs = error_incident.loc[error_incident['cmdb_id'] == cmdb, 'message'].dropna().unique()[:5]
    return " || ".join(map(str, msgs))

err_grp['sample_messages'] = err_grp['cmdb_id'].apply(sample_messages_for)

# top-5 by count
error_top5_summary = err_grp.sort_values('count', ascending=False).head(5).reset_index(drop=True)
error_top5_summary = error_top5_summary[['cmdb_id', 'count', 'earliest_error_epoch', 'earliest_error_local', 'sample_messages']]

# Final compact outputs (bounded)
log_thresholds.shape, log_incident.shape, log_anomaly_summary_top20, total_log_pairs_scanned, log_incident_rows, error_top5_summary

```
Out[4]:
```
```python
# Plain-English summary string of the log and error_log analyses (compact).
summary = (
    "Log analysis (incident window 2022-03-21 07:30:00 to 08:00:00 CST / UTC+8):\n"
    f"- Computed global thresholds for {log_thresholds.shape[0]} (cmdb_id, log_name) pairs using the full log series.\n"
    f"- Incident-window log rows: {log_incident_rows} rows across {total_log_pairs_scanned} distinct (cmdb_id, log_name) pairs.\n"
    "- Many log.error_count series show global_p95 == global_p5 == 0.0. As implemented, points equal to 0.0\n"
    "  satisfy both >=P95 and <=P5 checks, so they are counted in both high and low anomaly counters.\n"
    "  This causes reported totals like 60 anomalies (30 high + 30 low) for several services at the window start (07:30:00).\n\n"
    "Error log summary (incident window):\n"
    "- Top 3 cmdb_id by error count:\n"
    f"  1) frontend-0: count={error_top5_summary.loc[0,'count']}, earliest={error_top5_summary.loc[0,'earliest_error_local']}\n"
    f"     sample message(s): {error_top5_summary.loc[0,'sample_messages']}\n"
    f"  2) frontend-1: count={error_top5_summary.loc[1,'count']}, earliest={error_top5_summary.loc[1,'earliest_error_local']}\n"
    f"     sample message(s): {error_top5_summary.loc[1,'sample_messages']}\n"
    f"  3) frontend-2: count={error_top5_summary.loc[2,'count']}, earliest={error_top5_summary.loc[2,'earliest_error_local']}\n"
    f"     sample message(s): {error_top5_summary.loc[2,'sample_messages']}\n\n"
    "Notes and recommendations:\n"
    "- The anomaly counting logic using >= for high and <= for low produces double-counting when global P95==P5 (e.g., 0.0).\n"
    "  If this is undesired, change rules to strict > for high or < for low, or handle equality explicitly.\n"
    "- For next steps, inspect 'log_anomaly_summary_top20' and 'log_incident' for representative series, and\n"
    "  inspect 'error_top5_summary' for full sample messages and timestamps of frontend errors."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((62, 5), (1568, 11),                     cmdb_id         log_name  anomaly_count_p95_high  anomaly_count_p5_low  total_anomalies  earliest_anomaly_epoch earliest_anomaly_local  max_value_incident  min_value_incident  global_p95  global_p5
0               adservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
1               adservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
2               adservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
3             cartservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
4             cartservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
5             cartservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
6         currencyservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
7         currencyservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
8         currencyservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
9   recommendationservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
10        shippingservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
11        shippingservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
12  productcatalogservice-1  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
13  productcatalogservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
14  productcatalogservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
15  recommendationservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
16  recommendationservice-2  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
17        shippingservice-0  log.error_count                      30                    30               60              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
18           emailservice-2  log.error_count                      19                    19               38              1647819000    2022-03-21 07:30:00                 0.0                 0.0         0.0        0.0
19           emailservice-0  log.error_count                      19                    19               38              1647819060    2022-03-21 07:31:00                 0.0                 0.0         0.0        0.0, 62, 1568,       cmdb_id  count  earliest_error_epoch earliest_error_local                                    sample_messages
0  frontend-0    156            1647819005  2022-03-21 07:30:05  severity: warning, message: failed to retrieve...
1  frontend-1     77            1647819051  2022-03-21 07:30:51  severity: warning, message: failed to retrieve...
2  frontend-2     76            1647819039  2022-03-21 07:30:39  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse metric_anomaly_summary from kernel state
tz = pytz.timezone('Asia/Shanghai')

# 1) Aggregate by cmdb_id
# Ensure numeric columns are present and proper dtype
mas = metric_anomaly_summary.copy()
mas['total_anomalies'] = mas['total_anomalies'].fillna(0).astype(float)
mas['earliest_anomaly_epoch'] = pd.to_numeric(mas['earliest_anomaly_epoch'], errors='coerce')
mas['max_value_incident'] = pd.to_numeric(mas['max_value_incident'], errors='coerce')
mas['min_value_incident'] = pd.to_numeric(mas['min_value_incident'], errors='coerce')

agg_by_cmdb = (
    mas
    .groupby('cmdb_id', as_index=False)
    .agg(
        sum_total_anomalies = ('total_anomalies', 'sum'),
        num_kpis_with_anomaly = ('total_anomalies', lambda s: int((s > 0).sum())),
        earliest_anomaly_epoch = ('earliest_anomaly_epoch', 'min'),
        max_value_incident_over_kpis = ('max_value_incident', 'max'),
        min_value_incident_over_kpis = ('min_value_incident', 'min'),
    )
)

# Convert earliest epoch to int where present and add local string
def epoch_to_local_str(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg_by_cmdb['earliest_anomaly_epoch'] = agg_by_cmdb['earliest_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg_by_cmdb['earliest_anomaly_local'] = agg_by_cmdb['earliest_anomaly_epoch'].apply(lambda v: epoch_to_local_str(v) if not pd.isna(v) else '')

# Top 20 by sum_total_anomalies desc
metric_by_cmdb_top20 = agg_by_cmdb.sort_values('sum_total_anomalies', ascending=False).head(20).reset_index(drop=True)

# 2) Rows where kpi_name contains 'node-5' (case-sensitive), top 20 by total_anomalies desc
mask_node5 = mas['kpi_name'].str.contains('node-5', na=False)
metric_node5 = mas.loc[mask_node5, [
    'cmdb_id', 'kpi_name', 'total_anomalies', 'earliest_anomaly_local',
    'max_value_incident', 'min_value_incident', 'global_p95', 'global_p5'
]].copy()

metric_node5_top20 = metric_node5.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# 3) Total number of distinct cmdb_id values present in metric_anomaly_summary
distinct_cmdb_count = int(mas['cmdb_id'].nunique())

# Return compact results (variables to be displayed)
metric_by_cmdb_top20, metric_node5_top20, distinct_cmdb_count

```
Out[5]:
```
```python
# Compose a concise plain-English summary using existing kernel variables:
# metric_by_cmdb_top20, metric_node5_top20, distinct_cmdb_count

top_by_cmdb = metric_by_cmdb_top20
node5_rows = metric_node5_top20
n_cmdb = distinct_cmdb_count

# Build top-5 summary lines from metric_by_cmdb_top20
top5_lines = []
for i, row in top_by_cmdb.head(5).iterrows():
    top5_lines.append(
        f"{i+1}) {row['cmdb_id']}: sum_total_anomalies={int(row['sum_total_anomalies'])}, "
        f"num_kpis_with_anomaly={int(row['num_kpis_with_anomaly'])}, earliest={row['earliest_anomaly_local']}"
    )

# Build node-5 summary (show up to 5 examples)
node5_examples = []
for i, row in node5_rows.head(5).iterrows():
    node5_examples.append(
        f"{i+1}) {row['cmdb_id']} | {row['kpi_name']} | total_anomalies={int(row['total_anomalies'])} | earliest={row['earliest_anomaly_local']}"
    )

summary = (
    "Metric anomaly aggregation summary (incident window 2022-03-21 07:30:00 to 08:00:00 CST / UTC+8):\n\n"
    f"- Distinct cmdb_id values with KPI summaries: {n_cmdb}.\n\n"
    "- Top cmdb_id by total anomalies (top 5):\n" +
    "\n".join(top5_lines) + "\n\n"
    "- Observations:\n"
    "  * The highest totals are concentrated in services like 'adservice' and 'adservice2' (tens of thousands of anomaly counts),\n"
    "    and many frontend and core services (cartservice, productcatalogservice, checkoutservice) also rank highly.\n"
    "  * Earliest anomalies for the top cmdb_ids are at the start of the window: 2022-03-21 07:30:00 (UTC+8),\n"
    "    indicating issues began at or before the window start.\n\n"
    "- KPIs containing 'node-5' (examples, up to 5):\n" +
    ("\n".join(node5_examples) if node5_examples else "  (no node-5 KPIs found)") + "\n\n"
    "- Node-5 observation: multiple 'node-5' related KPIs (especially for shippingservice2-0 and some adservice entries)\n"
    "  show the maximum reported total_anomalies (often 60), with earliest occurrences at 07:30:00. Many of these KPIs\n"
    "  have identical global P95/P5 and identical incident values, which produced symmetric high/low counts.\n\n"
    "Conclusion: anomalies are widespread across many services, with heavy concentration in adservice/adservice2 and\n"
    "several frontend and core service components. Several node-5 related container KPIs stand out with maximum anomaly counts\n"
    "starting at the incident window outset. Inspect the detailed KPI and timeseries for these top cmdb_ids (variables:\n"
    "'metric_anomaly_summary', 'metric_by_cmdb_top20', 'metric_node5_top20') for root-cause investigation.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  sum_total_anomalies  num_kpis_with_anomaly  earliest_anomaly_epoch  max_value_incident_over_kpis  min_value_incident_over_kpis earliest_anomaly_local
0                  adservice              18380.0                    341              1647819000                  1.647523e+12                          -1.0    2022-03-21 07:30:00
1                 adservice2              16769.0                    333              1647819000                  1.647789e+12                          -1.0    2022-03-21 07:30:00
2                frontend2-0               6391.0                    176              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
3                 frontend-1               5872.0                    168              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
4                 frontend-0               5634.0                    164              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
5                 frontend-2               4976.0                    154              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
6          checkoutservice-0               4760.0                    130              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
7               adservice2-0               4622.0                    102              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
8    productcatalogservice-0               4518.0                    116              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
9    productcatalogservice-2               4476.0                    116              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
10             cartservice-1               4345.0                    100              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
11   productcatalogservice-1               4248.0                    112              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
12         checkoutservice-2               4236.0                    122              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
13         checkoutservice-1               4107.0                    118              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
14             cartservice-0               4070.0                     96              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
15            cartservice2-0               4044.0                     96              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
16               adservice-0               3963.0                     86              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
17               adservice-1               3825.0                     86              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
18  productcatalogservice2-0               3750.0                     96              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00
19        checkoutservice2-0               3740.0                    118              1647819000                  1.647821e+09                           0.0    2022-03-21 07:30:00,                cmdb_id                                           kpi_name  total_anomalies earliest_anomaly_local  max_value_incident  min_value_incident     global_p95      global_p5
0   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...             60.0    2022-03-21 07:30:00        65536.000000        65536.000000   65536.000000   65536.000000
1   shippingservice2-0             container.node-5.container_threads_max             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
2          adservice-0  container.node-5.container_cpu_cfs_throttled_p...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
3          adservice-0  container.node-5.container_cpu_cfs_throttled_s...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
4          adservice-0    container.node-5.container_cpu_load_average_10s             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
5   shippingservice2-0                 container.node-5.container_threads             60.0    2022-03-21 07:30:00           20.000000           20.000000      20.000000      20.000000
6   shippingservice2-0  container.node-5.container_tasks_state.uninter...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
7   shippingservice2-0     container.node-5.container_tasks_state.stopped             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
8   shippingservice2-0  container.node-5.container_fs_sector_writes./d...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
9   shippingservice2-0  container.node-5.container_fs_write_seconds./d...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
10  shippingservice2-0      container.node-5.container_fs_writes./dev/vda             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
11  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
12  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
13  shippingservice2-0  container.node-5.container_fs_writes_merged./d...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
14  shippingservice2-0  container.node-5.container_fs_io_time_weighted...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
15  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1             60.0    2022-03-21 07:30:00       604630.738281       604630.738281  604630.738281  604630.738281
16  shippingservice2-0  container.node-5.container_fs_read_seconds./de...             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
17  shippingservice2-0       container.node-5.container_fs_reads./dev/vda             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
18  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000
19  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda             60.0    2022-03-21 07:30:00            0.000000            0.000000       0.000000       0.000000, 59)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing kernel variables: metric_incident, metric_thresholds, metric_anomaly_summary, tz
# Ensure tz is set
tz = pytz.timezone('Asia/Shanghai')

# Filter incident rows whose kpi_name contains 'node-5' (case-sensitive)
mask_node5 = metric_incident['kpi_name'].str.contains('node-5', na=False)
node5_incident = metric_incident.loc[mask_node5].copy()

# If empty, create empty DataFrame with expected columns
if node5_incident.empty:
    cols = ['cmdb_id','kpi_name','total_anomalies','count_points_incident',
            'count_values_eq_global_p95','count_values_eq_global_p5','count_values_not_equal_thresholds',
            'earliest_neq_epoch','earliest_neq_local','sample_values','global_p95','global_p5']
    node5_kpi_summary = pd.DataFrame(columns=cols)
else:
    # Prepare a lookup for total_anomalies from metric_anomaly_summary
    ta_lookup = metric_anomaly_summary.set_index(['cmdb_id','kpi_name'])['total_anomalies'].to_dict()

    # Group and compute requested aggregates
    def summarize_group(g):
        cmdb = g.name[0]
        kpi = g.name[1]
        gp95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else np.nan
        gp5  = g['global_p5'].iloc[0] if 'global_p5' in g.columns else np.nan
        count_points_incident = int(len(g))
        # equality counts (exact match)
        count_eq_p95 = int((g['value'] == gp95).sum())
        count_eq_p5  = int((g['value'] == gp5).sum())
        count_not_eq = int(((g['value'] != gp95) & (g['value'] != gp5)).sum())
        # earliest not-equal epoch
        if count_not_eq > 0:
            earliest_neq_epoch = int(g.loc[(g['value'] != gp95) & (g['value'] != gp5), 'timestamp'].min())
            earliest_neq_local = pd.to_datetime(earliest_neq_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
        else:
            earliest_neq_epoch = np.nan
            earliest_neq_local = ''
        # sample values up to 3 distinct
        samp_vals = list(pd.Series(g['value'].dropna().unique())[:3])
        sample_values = " | ".join(map(lambda x: str(x), samp_vals))
        total_anomalies = float(ta_lookup.get((cmdb, kpi), 0.0))
        return pd.Series({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'total_anomalies': total_anomalies,
            'count_points_incident': count_points_incident,
            'count_values_eq_global_p95': count_eq_p95,
            'count_values_eq_global_p5': count_eq_p5,
            'count_values_not_equal_thresholds': count_not_eq,
            'earliest_neq_epoch': earliest_neq_epoch,
            'earliest_neq_local': earliest_neq_local,
            'sample_values': sample_values,
            'global_p95': gp95,
            'global_p5': gp5
        })

    node5_kpi_summary = (
        node5_incident
        .groupby(['cmdb_id', 'kpi_name'])
        .apply(summarize_group)
        .reset_index(drop=True)
    )

# Sort by total_anomalies desc and limit to top 20
node5_kpi_summary_top20 = node5_kpi_summary.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Expose the compact result
node5_kpi_summary_top20

```
Out[6]:
```
Summary of 'node-5' KPIs (inspection of incident-window metric rows vs global thresholds):

- What I checked: per (cmdb_id, kpi_name) entries whose kpi_name contains the substring "node-5" in the incident window.

- Main finding (top entries shown):
  - For the inspected KPIs (examples shown in the top-20), each had 30 incident points (count_points_incident = 30).
  - Each KPI shows total_anomalies = 60, with anomaly_count_p95_high = 30 and anomaly_count_p5_low = 30.
  - All 30 incident points equal the global P95 and also equal the global P5 (count_values_eq_global_p95 = 30, count_values_eq_global_p5 = 30).
  - There are zero points that differ from the thresholds (count_values_not_equal_thresholds = 0), so earliest_neq_epoch is null/empty.
  - Sample values equal the threshold values (examples: 65536, 0, 20, 604630.73828125 depending on KPI).

- Interpretation:
  - These "anomalies" are artifacts caused by equality to the global thresholds. Many KPIs have global_p95 == global_p5 (or incident values exactly equal both thresholds), and the anomaly logic used (value >= global_p95 counted as "high" and value <= global_p5 counted as "low") therefore counts the same points twice — once as high and once as low — producing total_anomalies = high + low (e.g., 30 + 30 = 60).
  - In short: the detected anomalies for node-5 KPIs are not actual deviations from typical behavior in the incident window but threshold-equality artifacts.

- Recommendation:
  - Change anomaly logic to avoid double-counting when thresholds are equal or when values equal thresholds. Options:
    - Use strict comparisons (high: value > global_p95, low: value < global_p5), or
    - Treat equality as belonging to only one side (e.g., equality assigns to low OR high, not both), or
    - Flag series with global_p95 == global_p5 for special handling/inspection.
  - Inspect the full time series for any KPI that truly departs from its historical distribution (if needed), but prioritize KPIs where count_values_not_equal_thresholds > 0.

If you want, I can (using the same variables) (a) list the full set of node-5 KPI rows that had any non-threshold-equal points, or (b) re-evaluate anomalies using strict > / < rules and produce an updated summary. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_anomalies  count_points_incident  count_values_eq_global_p95  count_values_eq_global_p5  count_values_not_equal_thresholds  earliest_neq_epoch earliest_neq_local    sample_values     global_p95      global_p5
0   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...             60.0                     30                          30                         30                                  0                 NaN                             65536.0   65536.000000   65536.000000
1   shippingservice2-0             container.node-5.container_threads_max             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
2          adservice-0  container.node-5.container_cpu_cfs_throttled_p...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
3          adservice-0  container.node-5.container_cpu_cfs_throttled_s...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
4          adservice-0    container.node-5.container_cpu_load_average_10s             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
5   shippingservice2-0                 container.node-5.container_threads             60.0                     30                          30                         30                                  0                 NaN                                20.0      20.000000      20.000000
6   shippingservice2-0  container.node-5.container_tasks_state.uninter...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
7   shippingservice2-0     container.node-5.container_tasks_state.stopped             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
8   shippingservice2-0  container.node-5.container_fs_sector_writes./d...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
9   shippingservice2-0  container.node-5.container_fs_write_seconds./d...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
10  shippingservice2-0      container.node-5.container_fs_writes./dev/vda             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
11  shippingservice2-0     container.node-5.container_fs_writes./dev/vda1             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
12  shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
13  shippingservice2-0  container.node-5.container_fs_writes_merged./d...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
14  shippingservice2-0  container.node-5.container_fs_io_time_weighted...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
15  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1             60.0                     30                          30                         30                                  0                 NaN                     604630.73828125  604630.738281  604630.738281
16  shippingservice2-0  container.node-5.container_fs_read_seconds./de...             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
17  shippingservice2-0       container.node-5.container_fs_reads./dev/vda             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
18  shippingservice2-0      container.node-5.container_fs_reads./dev/vda1             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000
19  shippingservice2-0    container.node-5.container_fs_reads_MB./dev/vda             60.0                     30                          30                         30                                  0                 NaN                                 0.0       0.000000       0.000000```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

# Reuse variables from kernel: metric_incident, metric_thresholds
tz = pytz.timezone('Asia/Shanghai')

# Ensure thresholds are present in metric_incident; if not, merge
if 'global_p95' not in metric_incident.columns or 'global_p5' not in metric_incident.columns:
    metric_incident = metric_incident.merge(
        metric_thresholds[['cmdb_id', 'kpi_name', 'global_p95', 'global_p5']],
        on=['cmdb_id', 'kpi_name'],
        how='left'
    )

# Strict anomaly booleans
metric_incident['anomaly_high_strict'] = metric_incident['value'] > metric_incident['global_p95']
metric_incident['anomaly_low_strict']  = metric_incident['value'] < metric_incident['global_p5']
metric_incident['any_strict'] = metric_incident['anomaly_high_strict'] | metric_incident['anomaly_low_strict']

# earliest strict anomaly epoch helper (NaN if none)
metric_incident['strict_anom_epoch'] = np.where(metric_incident['any_strict'], metric_incident['timestamp'], np.nan)

# Aggregate per (cmdb_id, kpi_name)
agg_strict = metric_incident.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count_points_incident = ('value', 'count'),
    anomaly_count_high_strict = ('anomaly_high_strict', 'sum'),
    anomaly_count_low_strict  = ('anomaly_low_strict', 'sum'),
    max_value_incident = ('value', 'max'),
    min_value_incident = ('value', 'min'),
    earliest_strict_anomaly_epoch = ('strict_anom_epoch', 'min')
)

# Ensure integer counts and compute total
agg_strict['anomaly_count_high_strict'] = agg_strict['anomaly_count_high_strict'].astype(int)
agg_strict['anomaly_count_low_strict'] = agg_strict['anomaly_count_low_strict'].astype(int)
agg_strict['total_anomalies_strict'] = agg_strict['anomaly_count_high_strict'] + agg_strict['anomaly_count_low_strict']

# Convert earliest epoch to int where present and add local string
def epoch_to_local(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg_strict['earliest_strict_anomaly_epoch'] = agg_strict['earliest_strict_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg_strict['earliest_strict_anomaly_local'] = agg_strict['earliest_strict_anomaly_epoch'].apply(lambda v: epoch_to_local(v) if not pd.isna(v) else '')

# Join global thresholds for final output
agg_strict = agg_strict.merge(metric_thresholds[['cmdb_id', 'kpi_name', 'global_p95', 'global_p5']],
                              on=['cmdb_id', 'kpi_name'], how='left')

# Select and order requested columns
metric_strict_summary = agg_strict[[
    'cmdb_id', 'kpi_name',
    'anomaly_count_high_strict', 'anomaly_count_low_strict', 'total_anomalies_strict',
    'earliest_strict_anomaly_epoch', 'earliest_strict_anomaly_local',
    'max_value_incident', 'min_value_incident',
    'global_p95', 'global_p5', 'count_points_incident'
]]

# Top-20 by total_anomalies_strict desc
metric_strict_summary_top20 = metric_strict_summary.sort_values('total_anomalies_strict', ascending=False).head(20).reset_index(drop=True)

# Total distinct pairs scanned
total_pairs_strict_scanned = int(metric_incident.groupby(['cmdb_id', 'kpi_name']).ngroups)

# Return compact results (DataFrame and count)
metric_strict_summary_top20, total_pairs_strict_scanned

```
Out[7]:
```
Summary of strict (> P95 and < P5) anomaly re-evaluation (incident window 07:30–08:00 CST):

- Scope: 4,934 distinct (cmdb_id, kpi_name) pairs from the incident window were scanned and re-evaluated using strict comparisons against the previously computed global P95 and P5.

- Effect of using strict rules: the large, double-counted anomaly totals (e.g., 60 = 30 high + 30 low) caused by values exactly equal to thresholds largely disappear. Under strict > / < rules most KPI anomaly counts drop markedly.

- Top results (compact observations):
  - The highest strict-anomaly KPIs are mostly from frontend-1 (many mesh/istio-related metrics) and a few node/system KPIs.
  - Typical top-row counts are small: many KPIs show anomaly_count_high_strict = 2 and anomaly_count_low_strict = 2, giving total_anomalies_strict = 4 (over 30 incident points).
  - Example earliest strict anomalies appear at the start of the window (2022-03-21 07:30:00 UTC+8) for most frontend KPIs; one system KPI (node-2 system.fs.inodes.total) has earliest strict anomaly at 07:31:00.
  - Incident window point counts per KPI remain 30 (the full 30 minutes), but only a few points per series exceed (>P95) or fall below (<P5).

- Interpretation:
  - Many previously reported anomalies were artifacts of equality to thresholds. With strict rules, true deviations are much less widespread.
  - The remaining strict anomalies are concentrated in frontend-1 mesh/istio metrics and a few node/system metrics — these are the best candidates for further investigation.

- Recommended next steps:
  - Inspect the time series for the top strict-anomaly KPIs (e.g., the frontend-1 istio request/response metrics and node-2 inodes) to understand the context and whether those >P95 / <P5 points are meaningful.
  - If desired, compute severity (e.g., magnitude above/below threshold) or examine surrounding minutes to confirm persistence.
  - Keep the strict comparison for anomaly detection to avoid double-counting artifacts, or adopt an explicit tie-breaking rule when global_p95 == global_p5.

If you want, I can output the top-N strict-anomaly table rows or plot representative series (text summaries) for the top candidates.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  anomaly_count_high_strict  anomaly_count_low_strict  total_anomalies_strict  earliest_strict_anomaly_epoch earliest_strict_anomaly_local  max_value_incident  min_value_incident    global_p95     global_p5  count_points_incident
0   frontend-1  mesh.source.frontend.adservice.istio_request_b...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        6.200000e+03        0.000000e+00  5.851250e+03  1.123750e+03                     30
1   frontend-1  mesh.source.frontend.adservice.istio_request_d...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        2.540000e-02        0.000000e+00  2.216125e-02  4.422500e-03                     30
2   frontend-1  mesh.source.frontend.adservice.istio_request_d...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.220000e-02        0.000000e+00  1.196375e-02  2.211250e-03                     30
3   frontend-1  mesh.source.frontend.adservice.istio_requests....                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        8.000000e+00        0.000000e+00  6.775000e+00  1.450000e+00                     30
4   frontend-1  mesh.source.frontend.adservice.istio_response_...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        2.730000e+03        0.000000e+00  2.374875e+03  4.540000e+02                     30
5   frontend-1  mesh.source.frontend.adservice.istio_response_...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        5.000000e+02        0.000000e+00  4.718750e+02  9.062500e+01                     30
6   frontend-1  mesh.source.frontend.adservice.istio_response_...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        8.000000e+00        0.000000e+00  6.775000e+00  1.450000e+00                     30
7       node-2                             system.fs.inodes.total                          2                         2                       4                   1.647819e+09           2022-03-21 07:31:00        4.932346e+09        4.925870e+09  4.932173e+09  4.928943e+09                     30
8   frontend-1  mesh.source.frontend.cartservice.istio_request...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.450000e+01        4.000000e+00  1.377500e+01  5.850000e+00                     30
9   frontend-1  mesh.source.frontend.cartservice.istio_request...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.450000e+01        4.000000e+00  1.377500e+01  5.850000e+00                     30
10  frontend-1  mesh.source.frontend.cartservice.istio_respons...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        2.515000e+03        6.200000e+02  2.452750e+03  1.003000e+03                     30
11  frontend-1  mesh.source.frontend.cartservice.istio_respons...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.450000e+01        4.000000e+00  1.377500e+01  5.850000e+00                     30
12  frontend-1  mesh.source.frontend.currencyservice.istio_req...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        9.067500e+04        2.402500e+04  7.998000e+04  3.154250e+04                     30
13  frontend-1  mesh.source.frontend.currencyservice.istio_req...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.368710e+01        1.942500e-02  6.552796e+00  2.540000e-02                     30
14  frontend-1  mesh.source.frontend.currencyservice.istio_req...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        5.750000e+01        1.550000e+01  5.177500e+01  2.012500e+01                     30
15  frontend-1  mesh.source.frontend.currencyservice.istio_req...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        5.750000e+01        1.550000e+01  5.177500e+01  2.012500e+01                     30
16  frontend-1  mesh.source.frontend.cartservice.istio_request...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        2.677500e+00        8.900000e-03  1.120444e+00  9.161250e-03                     30
17  frontend-1   container.node-5.container_fs_usage_MB./dev/vda1                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.051328e+02        1.047031e+02  1.051059e+02  1.047153e+02                     30
18  frontend-1               container.node-5.container_last_seen                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        1.647821e+09        1.647819e+09  1.647821e+09  1.647819e+09                     30
19  frontend-1  container.node-5.container_memory_failures.con...                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00        5.915000e+02        1.320000e+02  4.945750e+02  2.062500e+02                     30, 4934)```
```

In [ ]:
# In[8]:

import numpy as np
import pandas as pd
import pytz

# Reuse trace_incident and trace_thresholds from kernel state
tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamps are int
trace_incident['timestamp'] = trace_incident['timestamp'].astype(int)

# Merge global thresholds into incident rows if not already present
if 'global_p95' not in trace_incident.columns or 'global_p5' not in trace_incident.columns:
    trace_incident = trace_incident.merge(
        trace_thresholds[['cmdb_id', 'trace_name', 'global_p95', 'global_p5']],
        on=['cmdb_id', 'trace_name'],
        how='left'
    )

# Compute strict anomaly booleans
trace_incident['anomaly_high_strict'] = trace_incident['value'] > trace_incident['global_p95']
trace_incident['anomaly_low_strict']  = trace_incident['value'] < trace_incident['global_p5']
trace_incident['any_strict'] = trace_incident['anomaly_high_strict'] | trace_incident['anomaly_low_strict']

# Helper column for earliest strict anomaly epoch
trace_incident['strict_anom_epoch'] = np.where(trace_incident['any_strict'], trace_incident['timestamp'], np.nan)

# Aggregate per (cmdb_id, trace_name)
agg = trace_incident.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count_points_incident = ('value', 'count'),
    anomaly_count_high_strict = ('anomaly_high_strict', 'sum'),
    anomaly_count_low_strict  = ('anomaly_low_strict', 'sum'),
    max_value_incident = ('value', 'max'),
    min_value_incident = ('value', 'min'),
    earliest_strict_anomaly_epoch = ('strict_anom_epoch', 'min')
)

# Convert types and compute totals
agg['anomaly_count_high_strict'] = agg['anomaly_count_high_strict'].astype(int)
agg['anomaly_count_low_strict'] = agg['anomaly_count_low_strict'].astype(int)
agg['total_anomalies_strict'] = agg['anomaly_count_high_strict'] + agg['anomaly_count_low_strict']

# Convert earliest epoch to int where present and add local time string
def epoch_to_local_str(x):
    if pd.isna(x):
        return ''
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')

agg['earliest_strict_anomaly_epoch'] = agg['earliest_strict_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
agg['earliest_strict_anomaly_local'] = agg['earliest_strict_anomaly_epoch'].apply(lambda v: epoch_to_local_str(v) if not pd.isna(v) else '')

# Join thresholds for final columns
agg = agg.merge(trace_thresholds[['cmdb_id', 'trace_name', 'global_p95', 'global_p5']], on=['cmdb_id', 'trace_name'], how='left')

# Select and order requested columns
trace_strict_summary = agg[[
    'cmdb_id', 'trace_name',
    'anomaly_count_high_strict', 'anomaly_count_low_strict', 'total_anomalies_strict',
    'earliest_strict_anomaly_epoch', 'earliest_strict_anomaly_local',
    'max_value_incident', 'min_value_incident',
    'global_p95', 'global_p5'
]]

# Top-20 by total_anomalies_strict descending
trace_strict_summary_top20 = trace_strict_summary.sort_values('total_anomalies_strict', ascending=False).head(20).reset_index(drop=True)

# Total distinct pairs scanned
total_trace_pairs_strict_scanned = int(trace_incident.groupby(['cmdb_id', 'trace_name']).ngroups)

# Return compact outputs
trace_strict_summary_top20, total_trace_pairs_strict_scanned

```
Out[8]:
```
Trace strict-anomaly re-evaluation (strict >P95 and <P5) — concise findings:

- Scope: 1,240 distinct (cmdb_id, trace_name) pairs from the incident window were scanned and re-evaluated using strict comparisons against global P95/P5.

- Overall result: strict-rule anomaly counts are small and sparse. Top entries typically show:
  - anomaly_count_high_strict = 2, anomaly_count_low_strict = 2, total_anomalies_strict = 4 for many trace series.
  - This indicates only a few minutes (not tens of minutes) per series exceeded or fell below global thresholds when using strict comparisons.

- Top-ranked trace types and components (examples from the top-20):
  - recommendationservice2-0: trace.to_productcatalogservice-2.duration_p95 — total_strict_anomalies = 4, earliest strict anomaly 2022-03-21 07:39:00.
  - adservice2-0, adservice-2, productcatalogservice-1, frontend2-0, checkoutservice2-0 — many top items are duration_p95/duration_mean/row_count metrics across service-to-service traces (istio/mesh traces).
  - Several top rows are duration percentiles (p95) and means; some are row_count metrics (e.g., frontend2-0 row_count with max 287).

- Timing: earliest strict anomalies mostly occur near the start of or within the incident window (examples: 07:30:00, 07:31:00, 07:35:00, 07:39:00, etc.), depending on the metric.

- Interpretation:
  - Replacing >=/<= with strict >/< removed the equality-artefact double-counting seen earlier. The remaining strict anomalies are modest in count and concentrated in particular trace interactions (frontend→service, service→service).
  - These strict anomalies are better candidates for meaningful deviations than the large counts produced by equality-including logic.

- Recommended next steps:
  - Inspect time series for the top strict-anomaly traces (the listed trace_name/cmdb_id pairs) to judge impact (magnitude, persistence, and correlation across services).
  - Prioritize traces affecting frontend→backend paths and those with multiple strict anomaly points across services (possible propagation).
  - Keep strict comparison or otherwise define tie-breaking for equality to avoid false double-counting.

If you want, I can list the top N strict-anomaly trace rows (table) or extract the per-trace timestamps/values for the top candidates for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                        trace_name  anomaly_count_high_strict  anomaly_count_low_strict  total_anomalies_strict  earliest_strict_anomaly_epoch earliest_strict_anomaly_local  max_value_incident  min_value_incident  global_p95   global_p5
0   recommendationservice2-0     trace.to_productcatalogservice-2.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:39:00            0.000068            0.000026    0.000047    0.000027
1               adservice2-0               trace.from_frontend2-0.duration_p95                          2                         2                       4                   1.647819e+09           2022-03-21 07:35:00            0.000059            0.000024    0.000037    0.000024
2                adservice-2                trace.from_frontend-0.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:47:00            0.000027            0.000012    0.000023    0.000012
3                adservice-2               trace.from_frontend-2.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:31:00            0.000017            0.000011    0.000015    0.000011
4    productcatalogservice-1               trace.from_frontend-1.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:31:00            0.004459            0.003679    0.004405    0.003748
5    productcatalogservice-1  trace.from_recommendationservice-2.duration_mean                          2                         2                       4                   1.647820e+09           2022-03-21 07:43:00            0.000053            0.000021    0.000027    0.000021
6    productcatalogservice-1                trace.from_frontend-1.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:41:00            0.005836            0.004242    0.005681    0.004333
7    productcatalogservice-1               trace.from_frontend-2.duration_mean                          2                         2                       4                   1.647820e+09           2022-03-21 07:47:00            0.004539            0.003659    0.004438    0.003776
8    productcatalogservice-1                trace.from_frontend-2.duration_p95                          2                         2                       4                   1.647819e+09           2022-03-21 07:37:00            0.005850            0.004253    0.005696    0.004307
9    productcatalogservice-1  trace.from_recommendationservice-0.duration_mean                          2                         2                       4                   1.647820e+09           2022-03-21 07:44:00            0.000030            0.000021    0.000029    0.000021
10               frontend2-0          trace.to_currencyservice2-0.duration_p95                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00            0.007245            0.005322    0.006338    0.005335
11               frontend2-0             trace.to_currencyservice2-0.row_count                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00          287.000000           89.000000  271.550000  151.750000
12        checkoutservice2-0   trace.to_productcatalogservice2-0.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:38:00            0.004864            0.004093    0.004746    0.004251
13   productcatalogservice-1               trace.from_frontend-0.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00            0.004927            0.003749    0.004254    0.003871
14   productcatalogservice-1                trace.from_frontend-0.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:42:00            0.005898            0.004542    0.005785    0.004597
15               frontend2-0          trace.to_checkoutservice2-0.duration_p95                          2                         2                       4                   1.647819e+09           2022-03-21 07:33:00            0.136018            0.067465    0.097461    0.075172
16               frontend2-0         trace.to_currencyservice2-0.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00            0.003156            0.000901    0.001748    0.000932
17        checkoutservice2-0    trace.to_productcatalogservice2-0.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:43:00            0.005756            0.004408    0.005671    0.004470
18        checkoutservice2-0         trace.to_shippingservice2-0.duration_mean                          2                         2                       4                   1.647819e+09           2022-03-21 07:30:00            0.000083            0.000052    0.000062    0.000053
19        checkoutservice2-0          trace.to_currencyservice2-0.duration_p95                          2                         2                       4                   1.647820e+09           2022-03-21 07:41:00            0.010667            0.000113    0.007254    0.001084, 1240)```
```

In [ ]:
# In[9]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[9]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_incident and metric_thresholds from kernel state
tz = pytz.timezone('Asia/Shanghai')

# 1) Filter to exact frontend cmdb_ids present in metric_incident
frontend_ids = ['frontend-0','frontend-1','frontend-2','frontend2-0','frontend']
# Ensure thresholds are merged into metric_incident
if not {'global_p95','global_p5'}.issubset(metric_incident.columns):
    metric_incident = metric_incident.merge(
        metric_thresholds[['cmdb_id','kpi_name','global_p95','global_p5']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )

df_fe = metric_incident[metric_incident['cmdb_id'].isin(frontend_ids)].copy()

# If no frontend rows, create empty outputs
if df_fe.empty:
    per_cmdb_summary = pd.DataFrame(columns=[
        'cmdb_id','distinct_kpis_with_strict_anomaly','total_strict_anomaly_points',
        'earliest_strict_anomaly_epoch','earliest_strict_anomaly_local','total_incident_rows_for_cmdb'
    ])
    top_kpis_limited = pd.DataFrame(columns=['cmdb_id','kpi_name','strict_anomaly_point_count'])
else:
    # 2) Compute strict anomaly booleans per row
    df_fe['anomaly_high_strict'] = df_fe['value'] > df_fe['global_p95']
    df_fe['anomaly_low_strict']  = df_fe['value'] < df_fe['global_p5']
    df_fe['any_strict'] = df_fe['anomaly_high_strict'] | df_fe['anomaly_low_strict']
    df_fe['timestamp'] = df_fe['timestamp'].astype(int)

    # Per-cmdb: distinct_kpis_with_strict_anomaly
    kpi_strict_flags = (
        df_fe
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(strict_points_in_kpi = ('any_strict', 'sum'))
    )
    kpi_strict_flags['has_strict'] = kpi_strict_flags['strict_points_in_kpi'] > 0

    per_cmdb = (
        kpi_strict_flags
        .groupby('cmdb_id', as_index=False)
        .agg(distinct_kpis_with_strict_anomaly = ('has_strict', 'sum'))
    )

    # total_strict_anomaly_points and earliest strict anomaly epoch per cmdb
    strict_rows = df_fe[df_fe['any_strict']].copy()
    strict_count_by_cmdb = strict_rows.groupby('cmdb_id', as_index=False).agg(
        total_strict_anomaly_points = ('any_strict', 'sum'),
        earliest_strict_anomaly_epoch = ('timestamp', 'min')
    )
    # total incident rows per cmdb
    incident_rows_by_cmdb = df_fe.groupby('cmdb_id', as_index=False).agg(
        total_incident_rows_for_cmdb = ('timestamp', 'count')
    )

    # Merge per-cmdb aggregates
    per_cmdb_summary = per_cmdb.merge(strict_count_by_cmdb, on='cmdb_id', how='left').merge(
        incident_rows_by_cmdb, on='cmdb_id', how='left'
    )

    # Fill NaNs for cmdbs with no strict anomalies
    per_cmdb_summary['total_strict_anomaly_points'] = per_cmdb_summary['total_strict_anomaly_points'].fillna(0).astype(int)
    per_cmdb_summary['earliest_strict_anomaly_epoch'] = per_cmdb_summary['earliest_strict_anomaly_epoch'].apply(lambda v: int(v) if not pd.isna(v) else np.nan)
    per_cmdb_summary['earliest_strict_anomaly_local'] = per_cmdb_summary['earliest_strict_anomaly_epoch'].apply(
        lambda v: pd.to_datetime(int(v), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S') if not pd.isna(v) else ''
    )
    per_cmdb_summary['total_incident_rows_for_cmdb'] = per_cmdb_summary['total_incident_rows_for_cmdb'].fillna(0).astype(int)

    # Reorder columns
    per_cmdb_summary = per_cmdb_summary[[
        'cmdb_id','distinct_kpis_with_strict_anomaly','total_strict_anomaly_points',
        'earliest_strict_anomaly_epoch','earliest_strict_anomaly_local','total_incident_rows_for_cmdb'
    ]]

    # 3) For each cmdb_id, top 5 kpi_name by strict anomaly point count
    per_kpi_counts = kpi_strict_flags[['cmdb_id','kpi_name','strict_points_in_kpi']].copy()
    per_kpi_counts['strict_points_in_kpi'] = per_kpi_counts['strict_points_in_kpi'].astype(int)

    # For each cmdb, take top 5 kpis; then concatenate and limit overall to 20 rows to satisfy bounding
    top_kpis_list = []
    for cmdb in per_kpi_counts['cmdb_id'].unique():
        sub = per_kpi_counts[per_kpi_counts['cmdb_id'] == cmdb].sort_values('strict_points_in_kpi', ascending=False).head(5)
        top_kpis_list.append(sub)
    if top_kpis_list:
        top_kpis = pd.concat(top_kpis_list, ignore_index=True)
    else:
        top_kpis = pd.DataFrame(columns=['cmdb_id','kpi_name','strict_points_in_kpi'])

    top_kpis = top_kpis.rename(columns={'strict_points_in_kpi':'strict_anomaly_point_count'})

    # Limit overall rows to 20 for controller readability
    top_kpis_limited = top_kpis.sort_values(['strict_anomaly_point_count','cmdb_id'], ascending=[False,True]).head(20).reset_index(drop=True)

# Final outputs (compact)
per_cmdb_summary, top_kpis_limited

```
Out[10]:
```
```python
# Compose a concise plain-English summary using the frontend-focused results already in kernel:
# per_cmdb_summary and top_kpis_limited

per = per_cmdb_summary
topk = top_kpis_limited

lines = []
lines.append("Frontend strict-anomaly summary (strict >P95 or <P5, incident window 2022-03-21 07:30–08:00 CST / UTC+8):")
lines.append("")
lines.append("Per-cmdb_id summary:")
for _, r in per.iterrows():
    lines.append(
        f"- {r['cmdb_id']}: {int(r['distinct_kpis_with_strict_anomaly'])} distinct KPIs with strict anomalies, "
        f"{int(r['total_strict_anomaly_points'])} strict-anomaly points total, earliest at {r['earliest_strict_anomaly_local'] or 'N/A'}, "
        f"{int(r['total_incident_rows_for_cmdb'])} incident rows inspected."
    )

lines.append("")
lines.append("Top contributing KPIs (up to 5 per frontend, showing KPI and strict-anomaly point count):")
# group topk by cmdb_id for display
if not topk.empty:
    for cmdb in topk['cmdb_id'].unique():
        subset = topk[topk['cmdb_id'] == cmdb]
        kp_lines = "; ".join([f"{row['kpi_name']} ({int(row['strict_anomaly_point_count'])})" for _, row in subset.iterrows()])
        lines.append(f"- {cmdb}: {kp_lines}")
else:
    lines.append("- (no KPI strict-anomaly rows found)")

lines.append("")
lines.append("Interpretation and recommendations:")
lines.append("- Frontend nodes show multiple KPIs with strict anomalies, but counts per KPI are modest (many top KPIs report 4 strict-anomaly points).")
lines.append("- frontend2-0 and frontend-0 have the highest total strict-anomaly points among frontends (281 and 257 respectively).")
lines.append("- Earliest strict anomalies for all frontends occurred at the start of the incident window (2022-03-21 07:30:00 UTC+8).")
lines.append("- Action: prioritize inspection of the listed top KPIs (container CPU/memory and mesh/HTTP latency KPIs) for persistence and impact; these are the best candidates for root-cause follow-up.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id  distinct_kpis_with_strict_anomaly  total_strict_anomaly_points  earliest_strict_anomaly_epoch earliest_strict_anomaly_local  total_incident_rows_for_cmdb
0     frontend                                  2                            7                     1647819000           2022-03-21 07:30:00                           120
1   frontend-0                                 80                          257                     1647819000           2022-03-21 07:30:00                          4920
2   frontend-1                                 71                          250                     1647819000           2022-03-21 07:30:00                          5040
3   frontend-2                                 74                          243                     1647819000           2022-03-21 07:30:00                          4620
4  frontend2-0                                 77                          281                     1647819000           2022-03-21 07:30:00                          5280,         cmdb_id                                           kpi_name  strict_anomaly_point_count
0      frontend                                       app.http.mrt                           4
1    frontend-0         container.node-5.container_cpu_cfs_periods                           4
2    frontend-0               container.node-5.container_last_seen                           4
3    frontend-0  container.node-5.container_memory_failures.hie...                           4
4    frontend-0       container.node-5.container_cpu_usage_seconds                           4
5    frontend-0  container.node-5.container_memory_failures.con...                           4
6    frontend-1         container.node-5.container_cpu_cfs_periods                           4
7    frontend-1      container.node-5.container_cpu_system_seconds                           4
8    frontend-1       container.node-5.container_cpu_usage_seconds                           4
9    frontend-1  container.node-5.container_memory_failures.hie...                           4
10   frontend-1              container.node-5.container_memory_rss                           4
11   frontend-2         container.node-5.container_cpu_cfs_periods                           4
12   frontend-2       container.node-5.container_cpu_usage_seconds                           4
13   frontend-2        container.node-5.container_cpu_user_seconds                           4
14   frontend-2      container.node-5.container_cpu_system_seconds                           4
15   frontend-2   container.node-5.container_fs_usage_MB./dev/vda1                           4
16  frontend2-0         container.node-6.container_cpu_cfs_periods                           4
17  frontend2-0  container.node-6.container_cpu_cfs_throttled_s...                           4
18  frontend2-0       container.node-6.container_cpu_usage_seconds                           4
19  frontend2-0      container.node-6.container_cpu_system_seconds                           4)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse error_df from kernel state
tz = pytz.timezone('Asia/Shanghai')

# Parameters / window
frontend_ids = ['frontend-0','frontend-1','frontend-2','frontend2-0','frontend']
start_ts = 1647819000
end_ts = 1647820800

# Ensure timestamp int
error_df['timestamp'] = error_df['timestamp'].astype(int)

# Filter to frontends and incident window
err_fe = error_df[
    (error_df['cmdb_id'].isin(frontend_ids)) &
    (error_df['timestamp'] >= start_ts) &
    (error_df['timestamp'] < end_ts)
].copy()

# If empty, prepare empty outputs
if err_fe.empty:
    message_group_top20 = pd.DataFrame(columns=[
        'cmdb_id','message_text_trunc','count','earliest_error_epoch','earliest_error_local','sample_messages'
    ])
    overall_earliest = pd.DataFrame([{
        'overall_earliest_error_epoch': np.nan,
        'overall_earliest_error_local': ''
    }])
    per_cmdb_summary = pd.DataFrame(columns=['cmdb_id','count','earliest_error_epoch','earliest_error_local'])
else:
    # Group by exact message text and cmdb_id
    grp = (
        err_fe
        .groupby(['cmdb_id','message'], as_index=False)
        .agg(
            count=('timestamp','count'),
            earliest_error_epoch=('timestamp','min')
        )
    )

    # Add earliest local string
    grp['earliest_error_local'] = grp['earliest_error_epoch'].apply(
        lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    )

    # Truncate message_text to 200 chars for display
    grp['message_text_trunc'] = grp['message'].astype(str).str.slice(0,200)

    # Collect up to 5 distinct sample full messages per (cmdb_id, message)
    # Since grouping is by exact message, usually only one sample; still implement robustly
    def sample_msgs(row):
        cmdb = row['cmdb_id']
        msg = row['message']
        # Find up to 5 distinct full messages in err_fe that match exactly this message (will be just that message)
        samples = err_fe[(err_fe['cmdb_id']==cmdb) & (err_fe['message']==msg)]['message'].dropna().unique()[:5]
        # For compactness, truncate each sample to 200 chars
        samples_trunc = [s if len(s)<=200 else s[:200] + '...'( ) for s in samples] if False else [ (s if len(s)<=200 else s[:200]+'...') for s in samples ]
        return " || ".join(samples_trunc)

    # Vectorized sample_messages creation
    # To avoid slow row-wise apply over potentially many rows, build mapping of (cmdb,message)->samples
    unique_pairs = err_fe[['cmdb_id','message']].drop_duplicates()
    samples_map = {}
    for _, r in unique_pairs.iterrows():
        cmdb = r['cmdb_id']
        msg = r['message']
        samples = err_fe[(err_fe['cmdb_id']==cmdb) & (err_fe['message']==msg)]['message'].dropna().unique()[:5]
        samples_trunc = [s if len(s)<=200 else s[:200]+'...' for s in samples]
        samples_map[(cmdb,msg)] = " || ".join(samples_trunc)

    grp['sample_messages'] = grp.apply(lambda r: samples_map.get((r['cmdb_id'], r['message']), ''), axis=1)

    # Prepare top-20 messages by count desc
    message_group_top20 = grp.sort_values('count', ascending=False).head(20)[[
        'cmdb_id','message_text_trunc','count','earliest_error_epoch','earliest_error_local','sample_messages'
    ]].reset_index(drop=True)

    # Overall earliest error across these frontend rows
    overall_min_epoch = int(err_fe['timestamp'].min())
    overall_min_local = pd.to_datetime(overall_min_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    overall_earliest = pd.DataFrame([{
        'overall_earliest_error_epoch': overall_min_epoch,
        'overall_earliest_error_local': overall_min_local
    }])

    # Per-cmdb_id counts and earliest times
    per_cmdb = (
        err_fe
        .groupby('cmdb_id', as_index=False)
        .agg(count=('timestamp','count'), earliest_error_epoch=('timestamp','min'))
    )
    per_cmdb['earliest_error_local'] = per_cmdb['earliest_error_epoch'].apply(
        lambda e: pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    )
    per_cmdb_summary = per_cmdb[['cmdb_id','count','earliest_error_epoch','earliest_error_local']]

# Expose compact outputs
message_group_top20, overall_earliest, per_cmdb_summary

```
Out[11]:
```
Summary of frontend error logs during the incident window (2022-03-21 07:30:00 to 08:00:00 CST / UTC+8):

- Overall earliest frontend error:
  - Epoch: 1647819005 → 2022-03-21 07:30:05 (UTC+8).

- Top grouped messages (by exact text) for frontends (compact):
  1. frontend-0 — "severity: warning, message: failed to retrieve..." — count 156, earliest 2022-03-21 07:30:05
  2. frontend-1 — "severity: warning, message: failed to retrieve..." — count 77, earliest 2022-03-21 07:30:51
  3. frontend-2 — "severity: warning, message: failed to retrieve..." — count 76, earliest 2022-03-21 07:30:39
  - For each group the sample message(s) are identical/truncated to the same warning string shown above.

- Per-frontend summary (counts and earliest):
  - frontend-0: 156 error rows; earliest 2022-03-21 07:30:05
  - frontend-1: 77 error rows; earliest 2022-03-21 07:30:51
  - frontend-2: 76 error rows; earliest 2022-03-21 07:30:39

Interpretation:
- The error activity is concentrated in the frontend pods and dominated by repeated "failed to retrieve..." warning messages.
- Errors begin almost immediately after the incident window start (earliest at 07:30:05) and are most frequent on frontend-0.
- The grouped messages appear identical (no diverse variants shown), so this likely reflects a recurring issue or repeated failure condition rather than many distinct error types.

If you want, I can (a) show full untruncated sample messages for a chosen frontend, or (b) correlate these frontend errors with strict anomalies found earlier (metrics/traces) around the earliest timestamps. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                 message_text_trunc  count  earliest_error_epoch earliest_error_local                                    sample_messages
0  frontend-0  severity: warning, message: failed to retrieve...    156            1647819005  2022-03-21 07:30:05  severity: warning, message: failed to retrieve...
1  frontend-1  severity: warning, message: failed to retrieve...     77            1647819051  2022-03-21 07:30:51  severity: warning, message: failed to retrieve...
2  frontend-2  severity: warning, message: failed to retrieve...     76            1647819039  2022-03-21 07:30:39  severity: warning, message: failed to retrieve...,    overall_earliest_error_epoch overall_earliest_error_local
0                    1647819005          2022-03-21 07:30:05,       cmdb_id  count  earliest_error_epoch earliest_error_local
0  frontend-0    156            1647819005  2022-03-21 07:30:05
1  frontend-1     77            1647819051  2022-03-21 07:30:51
2  frontend-2     76            1647819039  2022-03-21 07:30:39)```
```

In [ ]:
# In[12]:

import numpy as np
import pandas as pd
import pytz

# Reuse variables from kernel: metric_incident, metric_thresholds, trace_incident, trace_thresholds, error_df
tz = pytz.timezone('Asia/Shanghai')

# Ensure thresholds present in metric_incident and trace_incident
if not {'global_p95','global_p5'}.issubset(metric_incident.columns):
    metric_incident = metric_incident.merge(
        metric_thresholds[['cmdb_id','kpi_name','global_p95','global_p5']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )

if not {'global_p95','global_p5'}.issubset(trace_incident.columns):
    trace_incident = trace_incident.merge(
        trace_thresholds[['cmdb_id','trace_name','global_p95','global_p5']],
        on=['cmdb_id','trace_name'],
        how='left'
    )

# Helper to compute strict-anomaly rows and deviation ratio for a DataFrame
def compute_earliest_strict(df, id_cols, name_col):
    df = df.copy()
    df['timestamp'] = df['timestamp'].astype(int)
    # strict anomaly booleans
    df['is_high_anomaly'] = df['value'] > df['global_p95']
    df['is_low_anomaly']  = df['value'] < df['global_p5']
    df['is_strict'] = df['is_high_anomaly'] | df['is_low_anomaly']
    df_strict = df[df['is_strict']].copy()
    if df_strict.empty:
        # return empty with expected columns
        cols = ['timestamp','timestamp_local'] + id_cols + [name_col,'value','global_p95','global_p5','is_high_anomaly','deviation_ratio']
        return pd.DataFrame(columns=cols)
    # choose threshold used per row
    df_strict['used_threshold'] = np.where(df_strict['is_high_anomaly'], df_strict['global_p95'], df_strict['global_p5'])
    # avoid divide-by-zero by adding tiny epsilon
    eps = 1e-12
    df_strict['deviation_ratio'] = (df_strict['value'] - df_strict['used_threshold']).abs() / (df_strict['used_threshold'].abs() + eps)
    # timestamp_local
    df_strict['timestamp_local'] = pd.to_datetime(df_strict['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S')
    # select and order columns
    select_cols = ['timestamp','timestamp_local'] + id_cols + [name_col,'value','global_p95','global_p5','is_high_anomaly','deviation_ratio']
    df_out = df_strict.loc[:, select_cols].sort_values('timestamp', ascending=True).reset_index(drop=True)
    # return up to 10 earliest rows (if many share same earliest timestamp, this will include the first up to 10)
    earliest_ts = df_out['timestamp'].min()
    # include rows with earliest timestamp; if fewer than 10, include next earliest until reaching 10
    topn = df_out.head(10)
    return topn

# 1) Metrics earliest strict anomalies
metrics_earliest_df = compute_earliest_strict(metric_incident, ['cmdb_id'], 'kpi_name')

# 2) Traces earliest strict anomalies
traces_earliest_df = compute_earliest_strict(trace_incident, ['cmdb_id'], 'trace_name')

# 3) Earliest frontend error log timestamp and cmdb_id from error_df (restrict to frontends and incident window)
frontend_ids = ['frontend-0','frontend-1','frontend-2','frontend2-0','frontend']
start_ts = 1647819000
end_ts = 1647820800

err = error_df.copy()
err['timestamp'] = err['timestamp'].astype(int)
err_fe = err[(err['cmdb_id'].isin(frontend_ids)) & (err['timestamp'] >= start_ts) & (err['timestamp'] < end_ts)].copy()

if err_fe.empty:
    earliest_frontend_error_summary = pd.DataFrame([{
        'earliest_error_epoch': np.nan,
        'earliest_error_local': '',
        'cmdb_ids_at_earliest': ''
    }])
else:
    min_epoch = int(err_fe['timestamp'].min())
    min_local = pd.to_datetime(min_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S')
    cmdbs_at_min = sorted(err_fe[err_fe['timestamp'] == min_epoch]['cmdb_id'].unique().tolist())
    earliest_frontend_error_summary = pd.DataFrame([{
        'earliest_error_epoch': min_epoch,
        'earliest_error_local': min_local,
        'cmdb_ids_at_earliest': ", ".join(cmdbs_at_min)
    }])

# Final compact outputs (bounded)
metrics_earliest_df, traces_earliest_df, earliest_frontend_error_summary

```
Out[12]:
```
Summary of earliest strict anomalies vs frontend error logs (all times UTC+8):

- Metrics (earliest strict anomalies):
  - Earliest strict-anomaly metric rows occur at 2022-03-21 07:30:00.
  - These earliest metric rows are from adservice (multiple JVM/runtime & GC KPIs). Most metric deviation_ratios are very small (values close to thresholds); a few metrics (e.g., Eden_Space) are flagged high but with small relative deviation (~0.0007).

- Traces (earliest strict anomalies):
  - Earliest strict-anomaly trace rows also occur at 2022-03-21 07:30:00.
  - Several are for frontend-0 (trace.from_root.* and trace.self.*) at 07:30:00 and show substantial deviations (large deviation_ratio for error_rate and several duration metrics), indicating notable trace-level degradation at that exact minute.

- Frontend error logs (earliest):
  - Earliest frontend error log: epoch 1647819005 → 2022-03-21 07:30:05 at frontend-0.

- Timeline and interpretation:
  - Strict anomalies in metrics and traces appear at 07:30:00 (the first minute of the incident window).
  - The earliest frontend error log appears ~5 seconds later at 07:30:05 (frontend-0).
  - Trace anomalies at 07:30:00 already show strong deviations for frontend-0 (durations and error_rate), while metric deviations at the same time (adservice JVM metrics) are present but have small relative magnitudes.
  - This suggests trace-level signals (frontend request durations/error rates) spike at 07:30:00 and are followed shortly by repeated frontend warning logs at ~07:30:05 — a plausible sequence for further root-cause investigation (inspect frontend traces and service interactions at 07:30:00, and correlated adservice metrics).

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp      timestamp_local    cmdb_id                                           kpi_name         value    global_p95     global_p5  is_high_anomaly  deviation_ratio
0  1647819000  2022-03-21 07:30:00  adservice                   runtime.java_lang_Runtime_Uptime  2.959496e+08  2.976026e+08  2.960366e+08            False         0.000294
1  1647819000  2022-03-21 07:30:00  adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  7.216069e+10  7.256405e+10  7.218210e+10            False         0.000297
2  1647819000  2022-03-21 07:30:00  adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  9.100233e+10  9.152955e+10  9.104780e+10            False         0.000499
3  1647819000  2022-03-21 07:30:00  adservice  runtime.java_lang_Threading_CurrentThreadUserTime  8.850167e+10  8.903150e+10  8.855904e+10            False         0.000648
4  1647819000  2022-03-21 07:30:00  adservice             runtime.jvm_gc_collection_seconds.Copy  1.258333e-02  1.354583e-02  1.283333e-02            False         0.019481
5  1647819000  2022-03-21 07:30:00  adservice         runtime.jvm_memory_pool_MB_used.Eden_Space  4.751411e+00  4.748145e+00  3.879651e+00             True         0.000688
6  1647819000  2022-03-21 07:30:00  adservice        runtime.jvm_memory_pool_MB_used.Tenured_Gen  1.584575e+01  1.585235e+01  1.584921e+01            False         0.000218
7  1647819000  2022-03-21 07:30:00  adservice  runtime.jvm_memory_pool_allocated_MB_total.Ede...  3.682643e+05  3.702278e+05  3.683673e+05            False         0.000280
8  1647819000  2022-03-21 07:30:00  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.661530e+07  1.662239e+07  1.661906e+07            False         0.000226
9  1647819000  2022-03-21 07:30:00  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.661493e+07  1.662238e+07  1.661883e+07            False         0.000235,     timestamp      timestamp_local         cmdb_id                                  trace_name       value  global_p95   global_p5  is_high_anomaly  deviation_ratio
0  1647819000  2022-03-21 07:30:00      frontend-1  trace.to_recommendationservice-1.row_count    1.000000    4.000000    1.450000            False     3.103448e-01
1  1647819000  2022-03-21 07:30:00      frontend-0            trace.to_cartservice-2.row_count    4.000000    9.000000    5.000000            False     2.000000e-01
2  1647819000  2022-03-21 07:30:00  emailservice-2   trace.from_checkoutservice-0.duration_p95    0.000203    0.000221    0.000204            False     2.946955e-03
3  1647819000  2022-03-21 07:30:00      frontend-0               trace.from_root.duration_mean    3.568021    1.210617    0.038268             True     1.947274e+00
4  1647819000  2022-03-21 07:30:00      frontend-0                trace.from_root.duration_p95    7.209463    3.997729    0.052600             True     8.033897e-01
5  1647819000  2022-03-21 07:30:00      frontend-0                  trace.from_root.error_rate    0.047619    0.000000    0.000000             True     4.761900e+10
6  1647819000  2022-03-21 07:30:00      frontend-0                    trace.self.duration_mean    0.595343    0.139557    0.003797             True     3.265961e+00
7  1647819000  2022-03-21 07:30:00      frontend-0                     trace.self.duration_p95    0.603439    0.334843    0.006427             True     8.021540e-01
8  1647819000  2022-03-21 07:30:00      frontend-0                       trace.self.error_rate    0.056075    0.011702    0.000000             True     3.791773e+00
9  1647819000  2022-03-21 07:30:00      frontend-0                        trace.self.row_count  107.000000  294.550000  195.200000            False     4.518443e-01,    earliest_error_epoch earliest_error_local cmdb_ids_at_earliest
0            1647819005  2022-03-21 07:30:05           frontend-0)```
```